In [1]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class PrefixNextBitPaddedDataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, label in samples:
            x = flatten_state(seq)
            y = 1 if label > 0 else 0
            self.data.append((x, y))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [2]:
def collate_pad(batch):
    """
    batch: list of (x, y) where x is [T_i] of 0/1 tokens
    returns:
      X: [B, T] int64 in {0,1,PAD_ID}
      Y: [B]     int64 in {0,1}
      M: [B, T]  bool   True where PAD
    """
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    padded, masks = [], []
    for x in xs:
        pad_len = max_len - x.size(0)
        if pad_len > 0:
            x_pad = torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)])
            m_pad = torch.cat([torch.zeros(x.size(0), dtype=torch.bool),
                               torch.ones(pad_len, dtype=torch.bool)])   # True = PAD
        else:
            x_pad = x
            m_pad = torch.zeros_like(x, dtype=torch.bool)
        padded.append(x_pad); masks.append(m_pad)

    X = torch.stack(padded)                   # [B, T]
    Y = torch.tensor(ys, dtype=torch.long)    # [B]
    M = torch.stack(masks)                    # [B, T] (True=PAD)
    return X, Y, M

In [3]:
import numpy as np
import xgboost as xgb
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

def build_csr_from_samples(samples, max_len=None):
    """
    Builds a CSR matrix with shape (N, max_len) containing 0/1 values.
    Missing (pads) are simply absent in the sparse matrix.
    """
    X_list = []
    y = []
    lengths = []

    for seq, label in samples:
        x = flatten_state(seq).numpy().astype(np.int8)   # 0/1, variable length
        X_list.append(x)
        y.append(1 if label > 0 else 0)
        lengths.append(len(x))

    if max_len is None:
        max_len = max(lengths)

    # Build CSR
    indptr = [0]
    indices = []
    data = []
    for x in X_list:
        L = min(len(x), max_len)
        # store all positions (including zeros is wasteful); store only ones
        ones = np.flatnonzero(x[:L] == 1)
        indices.extend(ones.tolist())
        data.extend([1] * len(ones))
        indptr.append(len(indices))

    X = csr_matrix((np.array(data, dtype=np.float32),
                    np.array(indices, dtype=np.int32),
                    np.array(indptr, dtype=np.int32)),
                   shape=(len(X_list), max_len))

    y = np.array(y, dtype=np.int32)
    return X, y, max_len

In [4]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

PAD_ID = 2

def build_xy_from_samples(samples):
    """
    samples: list of (seq, label) same as your torch dataset
    returns:
      X_list: list of 1D np arrays of 0/1 tokens (variable length)
      y:      np array of 0/1 labels
    """
    X_list, y = [], []
    for seq, label in samples:
        x = flatten_state(seq).numpy().astype(np.int64)     # 0/1 vector
        X_list.append(x)
        y.append(1 if label > 0 else 0)
    return X_list, np.array(y, dtype=np.int64)

def pad_X_list(X_list, max_len=None, pad_value=PAD_ID):
    """
    Pads variable-length vectors to [N, max_len] with pad_value.
    If max_len is None, uses max length in X_list.
    """
    if max_len is None:
        max_len = max(len(x) for x in X_list)
    X = np.full((len(X_list), max_len), pad_value, dtype=np.int64)
    for i, x in enumerate(X_list):
        L = min(len(x), max_len)
        X[i, :L] = x[:L]
    return X, max_len

def train_xgb_sparse_with_split_and_plots(
    samples,
    test_frac=0.2,
    test_samples=None,
    seed=42,
    num_round=300,
    early_stopping_rounds=30,
    max_len_cap=None,   # set e.g. 2048 to avoid one crazy-long sample exploding dim
):
    # split
    if test_samples is None:
        y_all = np.array([1 if lbl > 0 else 0 for _, lbl in samples], dtype=np.int32)
        train_samples, test_samples = train_test_split(
            samples, test_size=test_frac, random_state=seed, shuffle=True,
            stratify=y_all if len(np.unique(y_all)) > 1 else None
        )
        print(f"[split] train={len(train_samples)} test={len(test_samples)}")
    else:
        train_samples = samples
        print(f"[explicit] train={len(train_samples)} test={len(test_samples)}")

    # build sparse matrices with shared max_len
    Xtr_tmp, ytr, max_len_tr = build_csr_from_samples(train_samples, max_len=None)
    Xte_tmp, yte, max_len_te = build_csr_from_samples(test_samples, max_len=None)

    max_len = max(max_len_tr, max_len_te)
    if max_len_cap is not None:
        max_len = min(max_len, max_len_cap)

    Xtr, ytr, _ = build_csr_from_samples(train_samples, max_len=max_len)
    Xte, yte, _ = build_csr_from_samples(test_samples, max_len=max_len)

    dtrain = xgb.DMatrix(Xtr, label=ytr)
    dtest  = xgb.DMatrix(Xte, label=yte)

    params = {
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "auc"],
        "eta": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "min_child_weight": 1.0,
        "tree_method": "hist",
        "seed": seed,
    }

    evals_result = {}
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_round,
        evals=[(dtrain, "train"), (dtest, "test")],
        evals_result=evals_result,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=50,
    )

    p = booster.predict(dtest)
    pred = (p >= 0.5).astype(np.int32)

    acc = accuracy_score(yte, pred)
    auc = roc_auc_score(yte, p) if len(np.unique(yte)) > 1 else float("nan")
    print(f"Best iter: {booster.best_iteration} | Test acc: {acc:.4f} | Test AUC: {auc:.4f} | max_len={max_len}")

    # plots
    iters = np.arange(len(evals_result["train"]["logloss"]))
    plt.figure(figsize=(7,4))
    plt.plot(iters, evals_result["train"]["logloss"], label="train")
    plt.plot(iters, evals_result["test"]["logloss"], label="test")
    plt.xlabel("boosting round"); plt.ylabel("logloss"); plt.title("XGBoost (sparse) Logloss")
    plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(iters, evals_result["train"]["auc"], label="train")
    plt.plot(iters, evals_result["test"]["auc"], label="test")
    plt.xlabel("boosting round"); plt.ylabel("AUC"); plt.title("XGBoost (sparse) AUC")
    plt.legend(); plt.tight_layout(); plt.show()

    return booster, {"acc": acc, "auc": auc, "max_len": max_len, "evals_result": evals_result}

In [5]:
from sequence_generator import generate_dataset
import pickle

## Create dataset on other lengths and evaluate on new length

r = 11
original_dataset = []

with open('original_dataset_9.pkl', 'rb') as file:
    original_dataset.extend(pickle.load(file))

original_dataset = original_dataset[:100]

# with open('original_dataset_11.pkl', 'rb') as file:
#     test_dataset.extend(pickle.load(file))
# test_dataset = test_dataset[:500]

for item in original_dataset:
    if len(item[0][0]) <= 5:
        original_dataset.remove(item)

# for item in test_dataset:
#     if len(item[0][0]) <= 5:
#         test_dataset.remove(item)

count = [0, 0]
for item in original_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

booster, metrics = train_xgb_sparse_with_split_and_plots(
    original_dataset,
    test_frac=0.2,
    num_round=300,
    early_stopping_rounds=30,
    max_len_cap=4096,  # optional: prevents one outlier from exploding feature dim
)

: 

: 